# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR⁲ dataset using the `mlcroissant` library, following the Croissant schema standard and referencing all fields and entities by their `@id`. 

### Dataset Source
The dataset is described by a Croissant schema available at the following URL and includes multiple record sets about mass spectrometry protein data from human samples.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All metadata, record sets, fields, and columns are referenced using their `@id` property.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name:\n  {getattr(metadata, 'name', None)}\n\nDescription:\n  {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets (`@id`), and for each, list the available fields and their IDs.

_The Croissant schema may have multiple record sets describing different tables or data files. We'll enumerate them and their fields._

In [ ]:
print("Available record sets (referenced by their @id):")
record_sets = []
# Some datasets have one or more record sets; enumerate them.
for rset in metadata.record_sets:
    print(f"  @id: {rset['@id']} | Name: {rset.get('name', None)} | Description: {rset.get('description', None)}")
    record_sets.append(rset['@id'])
    # List the fields in the record set, referencing their @id
    print("  Fields:")
    for field in rset.get('fields', []):
        print(f"    - @id: {field['@id']} | Name: {field.get('name', '')} | DataType: {field.get('dataType', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the record set and field `@id` from the above overview.

In [ ]:
# Fill in the primary record set @id to extract. For this FAIR^2 dataset, 
# the most relevant data table is typically the main protein table.
# Replace <record_set_id> with the actual @id found in the dataset: e.g., 'https://sen.science/fileobject/protein_table', if present.

# If the dataset only contains one record set, we use it directly.
if len(record_sets) == 1:
    main_record_set_id = record_sets[0]
else:
    main_record_set_id = record_sets[0]  # You can change this after inspecting the printed result above.

df = pd.DataFrame(dataset.records(record_set=main_record_set_id))
print(f"Fields (columns) for record set @id '{main_record_set_id}':\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, or group by categorical fields. All field references use their `@id`.

_We'll select a numeric field, filter rows based on a threshold, normalize values, and group by a key attribute, if present._

In [ ]:
# Identify numeric fields from the field list
numeric_field_id = None
group_field_id = None
for rset in metadata.record_sets:
    if rset["@id"] == main_record_set_id:
        for field in rset["fields"]:
            # Using Croissant dataType, select first float or integer field
            dt = (field["dataType"] if "dataType" in field else "").lower()
            if not numeric_field_id and ('float' in dt or 'integer' in dt or 'number' in dt):
                numeric_field_id = field["@id"]
            # Select a group-by field (categorical/text)
            if not group_field_id and ('text' in dt or 'string' in dt or 'category' in dt):
                group_field_id = field["@id"]

if not numeric_field_id:
    print("No suitable numeric field found for analysis.")
else:
    print(f"Using numeric field: {{numeric_field_id}} for filtering and normalization.")
    threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 0

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped mean of numeric fields by '{group_field_id}':")
        display_cols = [numeric_field_id, norm_col] if norm_col in grouped_df else [numeric_field_id]
        print(grouped_df[display_cols].head())

## 5. Visualization
Visualize distributions of numeric fields and relationships between them using matplotlib or seaborn. All axis labels should refer to field `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If group_field_id exists, plot boxplot
if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading a Croissant-formatted dataset using `mlcroissant`, explored the dataset using the standard `@id` referencing of entities, performed basic filtering and normalization of protein abundance or measurement fields, grouped results by sample or protein, and visualized the data distribution. You can adapt this notebook to any Croissant schema-compliant dataset by changing the schema URL and referencing new record set/field `@id` values.